# Verify final recovered-source census and preserved code identities


All registered source addresses remain represented. Bindings and candidate container members are checked against frozen bytes; container membership does not establish biological source association. The earlier code files are preserved when documentation or reader portability changes their current hashes.

In [1]:
from pathlib import Path
import json, hashlib, ast
from starplast.provenance import SourceFile
root = Path.cwd()
out = root / 'results/source_recovery_final_2026_10_07_v2'
manifest = json.loads((out / 'manifest.json').read_text())
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
for path, digest in manifest['input_sha256'].items():
    assert sha(path) == digest, path
for name, digest in manifest['outputs'].items():
    assert sha(out / name) == digest, name
rows = json.loads((out / 'source_locations.json').read_text())
bindings = json.loads((out / 'bindings.json').read_text())
containers = json.loads((out / 'containers.json').read_text())
assert len(rows) == 162 and len(bindings) == 69 and len(containers) == 11
for value in bindings.values():
    SourceFile(**value).verify()
members = [member for container in containers.values() for member in container['members']]
for member in members:
    SourceFile(**{k: member[k] for k in ('path', 'sha256', 'bytes', 'role', 'url', 'association')}).verify()
assert len(members) == 538
{'registered': len(rows), 'verified_bindings': len(bindings), 'verified_container_candidates': len(members), 'unverified_member_association': True}

{'registered': 162, 'verified_bindings': 69, 'verified_container_candidates': 538, 'unverified_member_association': True}

In [2]:
preserved = []
for folder in (root / 'results').iterdir():
    if not folder.is_dir() or not (folder / 'code').is_dir():
        continue
    for archived in (folder / 'code').glob('*.py'):
        matches = []
        for mp in folder.glob('*manifest.json'):
            payload = json.loads(mp.read_text())
            for path, digest in payload.get('input_sha256', {}).items():
                if Path(path).name == archived.name and sha(archived) == digest:
                    matches.append(str(mp.relative_to(root)))
        if matches:
            preserved.append({'file': str(archived.relative_to(root)), 'matching_manifests': matches})
assert len(preserved) >= 16
for name in ('consolidate_source_recovery.py', 'finalize_source_recovery.py', 'retrieve_primary_table.py'):
    ast.parse((root / 'scripts' / name).read_text(), feature_version=(3, 10))
ast.parse((root / 'starplast/ground_truth.py').read_text(), feature_version=(3, 10))
{'preserved_code_identities': len(preserved), 'minimum_python_syntax': '3.10', 'numerical_changes': 'none'}

{'preserved_code_identities': 31, 'minimum_python_syntax': '3.10', 'numerical_changes': 'none'}